# Importation des bibliothèques

In [5]:
import requests
import time
import pandas as pd

# Constantes

In [6]:
# URL de l'API avec le jeu de données de l'inventaire immobilier de l'État
URL_API = "https://www.data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/inventaire-immobilier-de-letat/records"

# Output path for the data
OUTPUT_PATH = "../src/data/raw/donnees_immobilieres.parquet"

# Fonctions

In [7]:
def get_all_real_estate_data(limit=50, max_records=1000, url=URL_API):
    """
    Récupère l'ensemble des enregistrements de l'inventaire immobilier de l'État
    en gérant la pagination et en limitant le nombre total d'enregistrements.

    Args:
        limit (int): Le nombre d'enregistrements par page (par défaut 50).
        max_records (int): Le nombre maximum d'enregistrements à récupérer (par défaut 1000).

    Returns:
        list: Une liste contenant les enregistrements récupérés, jusqu'à la limite spécifiée.
    """
    all_records = []
    offset = 0
    page_number = 1

    url = url

    while True:
        print(f"Récupération de la page {page_number} (offset: {offset})...")

        # Arrêter si le nombre d'enregistrements récupérés dépasse la limite
        if len(all_records) >= max_records:
            print("Limite d'enregistrements atteinte. Arrêt de la récupération.")
            break

        params = {
            "limit": limit,
            "offset": offset,
        }

        try:
            response = requests.get(url, params=params)
            response.raise_for_status()  # Lève une exception pour les erreurs HTTP

            data = response.json()
            records = data.get("results", [])

            # Si la liste des enregistrements est vide, c'est la fin des données
            if not records:
                print("Toutes les pages ont été récupérées.")
                break

            all_records.extend(records)

            # Incrémenter l'offset pour la prochaine page
            offset += limit
            page_number += 1

            # Pause pour ne pas surcharger le serveur de l'API (bonne pratique)
            time.sleep(1)

        except requests.exceptions.RequestException as e:
            print(f"Une erreur est survenue lors de la requête : {e}")
            break

    # S'assurer que le nombre final ne dépasse pas la limite
    return all_records[:max_records]

# Application

In [8]:
# Exemple d'utilisation de la fonction pour récupérer les données
all_data = get_all_real_estate_data(limit=50, max_records=1000)

# Afficher quelques informations sur les données récupérées
if all_data:
    print(f"\nRécupération terminée. Nombre total d'enregistrements : {len(all_data)}")

Récupération de la page 1 (offset: 0)...
Récupération de la page 2 (offset: 50)...
Récupération de la page 3 (offset: 100)...
Récupération de la page 4 (offset: 150)...
Récupération de la page 5 (offset: 200)...
Récupération de la page 6 (offset: 250)...
Récupération de la page 7 (offset: 300)...
Récupération de la page 8 (offset: 350)...
Récupération de la page 9 (offset: 400)...
Récupération de la page 10 (offset: 450)...
Récupération de la page 11 (offset: 500)...
Récupération de la page 12 (offset: 550)...
Récupération de la page 13 (offset: 600)...
Récupération de la page 14 (offset: 650)...
Récupération de la page 15 (offset: 700)...
Récupération de la page 16 (offset: 750)...
Récupération de la page 17 (offset: 800)...
Récupération de la page 18 (offset: 850)...
Récupération de la page 19 (offset: 900)...
Récupération de la page 20 (offset: 950)...
Récupération de la page 21 (offset: 1000)...
Limite d'enregistrements atteinte. Arrêt de la récupération.

Récupération terminée. No

In [9]:
# Création d'un DataFrame à partir de la liste de dictionnaires
df = pd.DataFrame(all_data)

# Affichage du DataFrame avant la sauvegarde
df.head()

,region,ue,id,type,fonction,designation_batiment_terrain,dept,adresse,code_postal,ville,pays,ministere,date_inventaire
0,POITOU,178661,IG00100077264,Espace naturel,Terrain en friche,XYNTHIA17091POIRIER GENEVIEVE,17,21 R DU 14 JUILLET,17230,Charron,France,Ecologie - Développement durable - Energie - A...,2017-12
1,NORD-PAS DE CALAIS,132144,IG00100007106,Espace naturel,Terrain en friche,QUAI AU BOIS - Dunkerque,59,QU AU BOIS,59140,Dunkerque,France,Ecologie - Développement durable - Energie - A...,2017-03
2,ILE DE France,143844,IB00100024371,Bureau,Immeuble de bureaux,BATIMENT PRINCIPAL - QUAI D'ORSAY - MAE,75,37 Quai D ORSAY,75000,PARIS 07,France,Affaires étrangères et européennes.,2013-05
3,NORD-PAS DE CALAIS,145878,IB00100013660,Bureau,Hôtel des impôts,SIP-SIE,59,195 Rue DE ROUBAIX,59500,DOUAI,France,"Budget, comptes publics et fonction publique.",2013-05
4,PACA,135078,IG00100016428,Espace naturel,Terrain en friche,ANCIENNE BASE AERO NAVALE,13,Lieu-dit LA BASE ET LE MOULIN DE GO,13130,BERRE L ETANG,France,"Ecologie, développement et aménagement durables.",2013-05


In [10]:
# Enregistrement du DataFrame dans un fichier Parquet
df.to_parquet(OUTPUT_PATH, index=False)